# SmartIntake: Introduction and Schema Exploration

**Tutorial reference:** Part 1 of the SmartIntake Learning Tutorial

This notebook introduces the SmartIntake domain and helps you internalise the five compliance fields before you write any LLM code. Understanding the schema deeply is the foundation for everything that follows.

**NOTE**: Make sure you have gone over the domain document 

---

## 1. The Five Compliance Fields

SmartIntake extracts exactly five fields from every regulatory query. Each field has a fixed set of permissible values (enum) or a free-text constraint.

Read this table carefully. You will use these values throughout the entire project.

In [1]:
# Define the schema as Python dictionaries so we can reference it throughout the notebooks.
# These are not the Pydantic model yet — that comes in NB-04.
# This cell just makes the allowed values explicit and inspectable.

SCHEMA = {
    "query_type": [
        "complaint",
        "submission",
        "variation",
        "safety_signal",
        "label_update",
        "inspection",
        "general_enquiry",
    ],
    "regulation_ref": [
        "FDA_21CFR",
        "EMA_CTR",
        "ICH_E2A",
        "ICH_Q10",
        "CDSCO_NDC",
        "GxP_GMP",
        "GxP_GCP",
        "other",
    ],
    "product_area": [
        "oncology",
        "cardiovascular",
        "infectious_disease",
        "cmc",
        "clinical",
        "labelling",
        "general",
    ],
    "urgency": ["routine", "standard", "urgent", "critical"],
    "submitting_team": "<free text — team or function name, never an individual name>",
}

# Print a summary so it is easy to refer back to during development
for field, values in SCHEMA.items():
    print(f"{field}:")
    if isinstance(values, list):
        for v in values:
            print(f"   {v}")
    else:
        print(f"   {values}")
    print()

query_type:
   complaint
   submission
   variation
   safety_signal
   label_update
   inspection
   general_enquiry

regulation_ref:
   FDA_21CFR
   EMA_CTR
   ICH_E2A
   ICH_Q10
   CDSCO_NDC
   GxP_GMP
   GxP_GCP
   other

product_area:
   oncology
   cardiovascular
   infectious_disease
   cmc
   clinical
   labelling
   general

urgency:
   routine
   standard
   urgent
   critical

submitting_team:
   <free text — team or function name, never an individual name>



## 2. Sample Test Inputs

The specification provides five sample test inputs. Before you build anything, read each one and try to determine the expected extraction manually.

This exercise builds the domain intuition your prompt engineering will rely on.

In [2]:
# The five canonical test inputs from the specification.
# These are used in every notebook to validate system behaviour.

TEST_INPUTS = {
    "T1": (
        "PV team here. We have a serious unexpected SUSAR for study NCT-2244 "
        "and need to report to EMA within 15 days per ICH E2A."
    ),
    "T2": (
        "Hi, we need to file a Type II variation for our cardiovascular product with the EMA."
        # Note: submitting_team and urgency are deliberately absent
    ),
    "T3": (
        "CMC Regulatory here. FDA issued a 483 observation on our manufacturing site in Pune. "
        "We have 15 business days to respond."
    ),
    "T4": "Can you check what the weather is like in Mumbai today?",
    "T5": (
        "Labelling team. We need to update the SmPC for our oncology product "
        "following the latest EMA assessment report."
    ),
}

for key, text in TEST_INPUTS.items():
    print(f"{key}: {text}")
    print()

T1: PV team here. We have a serious unexpected SUSAR for study NCT-2244 and need to report to EMA within 15 days per ICH E2A.

T2: Hi, we need to file a Type II variation for our cardiovascular product with the EMA.

T3: CMC Regulatory here. FDA issued a 483 observation on our manufacturing site in Pune. We have 15 business days to respond.

T4: Can you check what the weather is like in Mumbai today?

T5: Labelling team. We need to update the SmPC for our oncology product following the latest EMA assessment report.



## 3. Manual Prediction Exercise

Before running any LLM, fill in your predicted extractions for each test input.

This is the baseline you will compare against your model's output in NB-03 and NB-04.

In [3]:
# Fill in your predictions here.
# Use the exact enum values from SCHEMA above.
# Use None for any field you think the model will need to ask for.

PREDICTED_EXTRACTIONS = {
    "T1": {
        "query_type": None,          # YOUR PREDICTION
        "regulation_ref": None,      # YOUR PREDICTION
        "product_area": None,        # YOUR PREDICTION
        "urgency": None,             # YOUR PREDICTION
        "submitting_team": None,     # YOUR PREDICTION
    },
    "T2": {
        "query_type": None,
        "regulation_ref": None,
        "product_area": None,
        "urgency": None,
        "submitting_team": None,
    },
    "T3": {
        "query_type": None,
        "regulation_ref": None,
        "product_area": None,
        "urgency": None,
        "submitting_team": None,
    },
    "T4": "FALLBACK",  # This should not extract any fields
    "T5": {
        "query_type": None,
        "regulation_ref": None,
        "product_area": None,
        "urgency": None,
        "submitting_team": None,
    },
}

print("Your predictions are saved. Compare them against model output in later notebooks.")

Your predictions are saved. Compare them against model output in later notebooks.


## 4. The Compliance Constraint

SmartIntake operates in a regulated pharmaceutical environment. Two constraints shape every design decision:

1. **Log safety:** logs must never contain sensitive identifiers or free-text user messages.
2. **Audit traceability:** every record must be attributable to a submitting team.

The cell below demonstrates what a compliant log entry looks like versus a non-compliant one.

In [4]:
import json
from datetime import datetime

# COMPLIANT log entry — structured fields only, no raw message
compliant_log_entry = {
    "timestamp": datetime.now().isoformat(),
    "event": "extraction_attempt",
    "prompt_tokens": 312,
    "completion_tokens": 87,
    "fields_extracted": ["query_type", "regulation_ref", "product_area"],
    "fields_missing": ["urgency", "submitting_team"],
}

# NON-COMPLIANT log entry — raw user message included
non_compliant_log_entry = {
    "timestamp": datetime.now().isoformat(),
    "event": "extraction_attempt",
    "user_message": "PV team here. SUSAR for NCT-2244...",  # <-- VIOLATION
    "prompt_tokens": 312,
    "completion_tokens": 87,
}

print("COMPLIANT log entry:")
print(json.dumps(compliant_log_entry, indent=2))
print()
print("NON-COMPLIANT log entry — the user_message field must never appear in logs:")
print(json.dumps(non_compliant_log_entry, indent=2))

COMPLIANT log entry:
{
  "timestamp": "2026-06-24T21:20:29.524711",
  "event": "extraction_attempt",
  "prompt_tokens": 312,
  "completion_tokens": 87,
  "fields_extracted": [
    "query_type",
    "regulation_ref",
    "product_area"
  ],
  "fields_missing": [
    "urgency",
    "submitting_team"
  ]
}

NON-COMPLIANT log entry — the user_message field must never appear in logs:
{
  "timestamp": "2026-06-24T21:20:29.524757",
  "event": "extraction_attempt",
  "user_message": "PV team here. SUSAR for NCT-2244...",
  "prompt_tokens": 312,
  "completion_tokens": 87
}


---
## Summary

You have:
- Reviewed the five compliance fields and their permissible values
- Read all five test inputs and formed predictions about the expected extractions
- Understood the compliance constraint that shapes the logging and output requirements

**Next:** Open `NB-02_api_calls.ipynb` to make your first authenticated LLM API call.